In [2]:
import os
import numpy as np
import pandas as pd
import mne
from scipy import signal
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
mne.set_log_level('ERROR')

In [3]:
# Rutas
DATA_DIR = '/home/manu/TFG2/HBN_EEG'
OUTPUT_DIR = '/home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_HBN'

# Parámetros
FS = 500  # Frecuencia
FREQ_BAND = (8, 12)  # Banda Alpha
WINDOW_DURATION = 2.0  # segundos
WINDOW_SAMPLES = int(WINDOW_DURATION * FS)  # 1000 muestras
MAX_WINDOWS_PER_SUBJECT = 30  # Limitar para no generar millones

print(f"Ventana: {WINDOW_DURATION}s = {WINDOW_SAMPLES} samples")
print(f"Banda: {FREQ_BAND[0]}-{FREQ_BAND[1]} Hz")
print(f"Max ventanas por sujeto: {MAX_WINDOWS_PER_SUBJECT}")

Ventana: 2.0s = 1000 samples
Banda: 8-12 Hz
Max ventanas por sujeto: 30


In [4]:
# Leer participants
participants = pd.read_csv(os.path.join(DATA_DIR, 'participants.tsv'), sep='\t')
print(f"Total participantes: {len(participants)}")
print(f"\nDistribución del factor attention:")
print(participants['attention'].describe())

Total participantes: 136

Distribución del factor attention:
count    132.000000
mean      -0.048576
std        0.887195
min       -1.786000
25%       -0.775000
50%       -0.144500
75%        0.659000
max        2.406000
Name: attention, dtype: float64


In [5]:
# Filtrar solo los que tienen Resting State disponible
if 'RestingState' in participants.columns:
    participants_valid = participants[participants['RestingState'] == 'available'].copy()
else:
    participants_valid = participants.copy()

# Eliminar los que no tengan attention
participants_valid = participants_valid.dropna(subset=['attention'])

print(f"Participantes válidos: {len(participants_valid)}")

# Binarizar attention en la mediana
median_attention = participants_valid['attention'].median()
participants_valid['label'] = (participants_valid['attention'] > median_attention).astype(int)

print(f"\nMediana de attention: {median_attention:.3f}")
print(f"Low attention (0):  {(participants_valid['label']==0).sum()}")
print(f"High attention (1): {(participants_valid['label']==1).sum()}")

Participantes válidos: 132

Mediana de attention: -0.145
Low attention (0):  66
High attention (1): 66


In [6]:
np.random.seed(42)

# Separar por clase
low_subjects = participants_valid[participants_valid['label']==0]['participant_id'].tolist()
high_subjects = participants_valid[participants_valid['label']==1]['participant_id'].tolist()

# Shuffle
np.random.shuffle(low_subjects)
np.random.shuffle(high_subjects)

# Split 80/20
low_split = int(0.8 * len(low_subjects))
high_split = int(0.8 * len(high_subjects))

train_low = low_subjects[:low_split]
train_high = high_subjects[:high_split]
test_low = low_subjects[low_split:]
test_high = high_subjects[high_split:]

print(f"Training: {len(train_low)} Low + {len(train_high)} High = {len(train_low)+len(train_high)}")
print(f"Test:     {len(test_low)} Low + {len(test_high)} High = {len(test_low)+len(test_high)}")

Training: 52 Low + 52 High = 104
Test:     14 Low + 14 High = 28


In [7]:
def compute_coherence_matrix(data, fs, freq_band):
    """Matriz de coherencia cuadrada."""
    n_channels = data.shape[0]
    coh_matrix = np.zeros((n_channels, n_channels))
    
    for i in range(n_channels):
        for j in range(i, n_channels):
            if i == j:
                coh_matrix[i, j] = 1.0
            else:
                f, Cxy = signal.coherence(data[i], data[j], fs=fs, nperseg=min(256, len(data[i])))
                freq_mask = (f >= freq_band[0]) & (f <= freq_band[1])
                coh_value = np.mean(Cxy[freq_mask]) if np.any(freq_mask) else 0.0
                coh_matrix[i, j] = coh_value
                coh_matrix[j, i] = coh_value
    return coh_matrix


def select_channels(n_target, total_channels):
    """Selecciona canales equiespaciados."""
    if n_target >= total_channels:
        return list(range(total_channels))
    indices = np.linspace(0, total_channels - 1, n_target, dtype=int)
    return indices.tolist()


def process_subject(subject_id):
    """Procesa un sujeto y devuelve sus matrices."""
    eeg_file = os.path.join(
        DATA_DIR, subject_id, 'eeg',
        f'{subject_id}_task-RestingState_eeg.set'
    )
    
    if not os.path.exists(eeg_file):
        return []
    
    try:
        raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose=False)
        
        # Coger solo los 128 canales EEG (E1-E128), descartar Cz si está
        eeg_channels = [ch for ch in raw.ch_names if ch.startswith('E')][:128]
        raw.pick_channels(eeg_channels)
        
        # Filtrar en banda alpha (mejora la calidad)
        raw.filter(l_freq=1, h_freq=40, verbose=False)
        
        data = raw.get_data()  # (128, n_samples)
        n_samples = data.shape[1]
        
        if data.shape[0] < 128:
            return []
        
        # Segmentar en ventanas no solapadas
        matrices = []
        n_windows = (n_samples - WINDOW_SAMPLES) // WINDOW_SAMPLES
        
        # Limitar número de ventanas
        if n_windows > MAX_WINDOWS_PER_SUBJECT:
            # Espaciar uniformemente
            window_indices = np.linspace(0, n_windows-1, MAX_WINDOWS_PER_SUBJECT, dtype=int)
        else:
            window_indices = range(n_windows)
        
        for w_idx in window_indices:
            start = w_idx * WINDOW_SAMPLES
            end = start + WINDOW_SAMPLES
            window = data[:, start:end]
            mat = compute_coherence_matrix(window, FS, FREQ_BAND)
            matrices.append(mat)
        
        return matrices
    
    except Exception as e:
        # print(f"  Error en {subject_id}: {e}")
        return []

In [11]:
for split in ['Training', 'Test']:
    for n_channels in ['8', '16', '32', '64']:
        for label in ['low', 'high']:
            path = os.path.join(OUTPUT_DIR, split, n_channels, label)
            os.makedirs(path, exist_ok=True)

print(f"Estructura creada en: {OUTPUT_DIR}")

Estructura creada en: /home/manu/TFG2/EEG-Gold-Standard-main/ConnectivityMatrices_HBN


In [8]:
def process_and_save(subjects, label_str, split_name):
    """Procesa lista de sujetos y guarda matrices."""
    counts = {ch: 0 for ch in [8, 16, 32, 64]}
    
    for subject_id in tqdm(subjects, desc=f"{split_name}/{label_str}"):
        matrices_128 = process_subject(subject_id)
        
        if len(matrices_128) == 0:
            continue
        
        for n_ch in [8, 16, 32, 64]:
            ch_subset = select_channels(n_ch, 128)
            
            for idx, mat in enumerate(matrices_128):
                sub_mat = mat[np.ix_(ch_subset, ch_subset)]
                fname = f"{subject_id}_w{idx:03d}.npy"
                path = os.path.join(OUTPUT_DIR, split_name, str(n_ch), label_str, fname)
                np.save(path, sub_mat)
                counts[n_ch] += 1
    
    return counts

In [14]:
# Generar Training
print("="*60)
print("TRAINING")
print("="*60)
train_low_counts = process_and_save(train_low, 'low', 'Training')
train_high_counts = process_and_save(train_high, 'high', 'Training')

TRAINING


Training/low:   0%|          | 0/52 [00:00<?, ?it/s]

Training/high: 100%|██████████| 52/52 [5:37:52<00:00, 389.85s/it]  


In [9]:
# Generar Test
print("="*60)
print("TEST")
print("="*60)
test_low_counts = process_and_save(test_low, 'low', 'Test')
test_high_counts = process_and_save(test_high, 'high', 'Test')

TEST


Test/low:   0%|          | 0/14 [00:00<?, ?it/s]

Test/high: 100%|██████████| 14/14 [1:06:57<00:00, 286.96s/it]


In [10]:
print("="*60)
print("RESUMEN FINAL - HBN EEG")
print("="*60)

print(f"\nSujetos Training: {len(train_low) + len(train_high)} ({len(train_low)} Low + {len(train_high)} High)")
print(f"Sujetos Test: {len(test_low) + len(test_high)} ({len(test_low)} Low + {len(test_high)} High)")

print("\n--- TRAINING ---")
for ch in [64, 32, 16, 8]:
    total = train_low_counts[ch] + train_high_counts[ch]
    print(f"{ch} canales: Low={train_low_counts[ch]}, High={train_high_counts[ch]}, Total={total}")

print("\n--- TEST ---")
for ch in [64, 32, 16, 8]:
    total = test_low_counts[ch] + test_high_counts[ch]
    print(f"{ch} canales: Low={test_low_counts[ch]}, High={test_high_counts[ch]}, Total={total}")

print("\n" + "="*60)
print(f"Matrices en: {OUTPUT_DIR}")
print("="*60)

RESUMEN FINAL - HBN EEG

Sujetos Training: 104 (52 Low + 52 High)
Sujetos Test: 28 (14 Low + 14 High)

--- TRAINING ---


NameError: name 'train_low_counts' is not defined